# Action Recognition
 Foundations of Machine Learning SS26    
 Group 11 - G. Sammet, M. Schlichting
### Overall Goal
- Classify 4 actions
### Dataset 
- RGB videos
    - ca 2 seconds long
    - resolution 1920x1080
- ca. 900 videos per action
- 4 actions: waving, capitulate, cross arms and clapping

    **Aquisition:**    
    We contacted ROSE Lab to get permission for the download. Due to hardware limits we downloaded the zip files in batched and ran the ``process_NTU.py`` to extract only the action classes we wanted.

    *The research in this project used the NTU RGB+D (or NTU RGB+D 120) Action Recognition Dataset made available by the ROSE Lab at the Nanyang Technological University, Singapore.*

    ROSE Lab. Action recognition datasets: “NTU RGB+D” dataset and “NTU RGB+D 120” dataset [Dataset]. https://rose1.ntu.edu.sg/dataset/actionRecognition/  

### Imports

In [44]:
from pathlib import Path
import random
import shutil

import torch
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Experiment 1: Simple One-Frame-Model
single frame -> pre-trained ResNet18 -> custom classifier output layer

### Load ResNet18 Model & Dataset

In [45]:
# load pretrained model
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)

# modify last layer to 4 classes
resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, 4)

# define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet_model.fc.parameters(), lr=0.001, momentum=0.9)

# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [50]:
# load data and transform
output_dir = Path("data/three_quarter_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_three_quarter = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_three_quarter = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_three_quarter = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Training

In [47]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # Print the results for the current epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [51]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model_three_quarter = resnet_model.to(device)
train(model_three_quarter, train_loader_three_quarter, val_loader_three_quarter, criterion, optimizer, num_epochs=10)

Epoch [1/10], train loss: 1.1870, train acc: 0.4543, val loss: 1.0077, val acc: 0.5298
Epoch [2/10], train loss: 0.9262, train acc: 0.5926, val loss: 0.8766, val acc: 0.6158
Epoch [3/10], train loss: 0.8492, train acc: 0.6336, val loss: 0.8464, val acc: 0.6140
Epoch [4/10], train loss: 0.7957, train acc: 0.6655, val loss: 0.7919, val acc: 0.6491
Epoch [5/10], train loss: 0.7619, train acc: 0.6840, val loss: 0.7812, val acc: 0.6439
Epoch [6/10], train loss: 0.7313, train acc: 0.6892, val loss: 0.7403, val acc: 0.6860
Epoch [7/10], train loss: 0.7186, train acc: 0.6896, val loss: 0.7396, val acc: 0.6596
Epoch [8/10], train loss: 0.7103, train acc: 0.6960, val loss: 0.7517, val acc: 0.6561
Epoch [9/10], train loss: 0.7104, train acc: 0.6836, val loss: 0.7395, val acc: 0.6649
Epoch [10/10], train loss: 0.6979, train acc: 0.7016, val loss: 0.7114, val acc: 0.6807


# TODO: EVAL FUNCTION

In [ ]:
def evaluate_model():
    pass

## Experiment 2: 3-Model-3-Frames-Approach
3 frames per video -> one model per frame type -> average over class predictions -> classification

In [ ]:
# load datat for middle frame
output_dir = Path("data/middle_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_middle = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_middle = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_middle = DataLoader(test_dataset, batch_size=32, shuffle=False)

# train model 2
model_middle = resnet_model.to(device)
train(model_middle, train_loader_middle, val_loader_middle, criterion, optimizer, num_epochs=10)

In [ ]:
evaluate_model(model_middle, test_loader_middle, device)

In [ ]:
# load datat for end frame
output_dir = Path("data/end_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_end = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_end = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_end = DataLoader(test_dataset, batch_size=32, shuffle=False)

# train model 2
model_end = resnet_model.to(device)
train(model_end, train_loader_end, val_loader_end, criterion, optimizer, num_epochs=10)